# 使用 Assistants API（GPT-4）和 DALL·E-3 创建幻灯片

本 notebook 演示了如何使用新的 [Assistants API](https://platform.openai.com/docs/assistants/overview)（GPT-4）和 DALL·E-3 制作内容翔实、视觉美观的幻灯片。<br>
制作幻灯片是许多工作的关键环节，但可能既繁琐又耗时。此外，从数据中提炼洞察并在幻灯片上有效表达出来也颇具挑战。<br><br> 本 cookbook 示例将演示如何利用新的 Assistants API 为你完成端到端的幻灯片创建流程，你无需打开 Microsoft PowerPoint 或 Google Slides，从而节省宝贵的时间和精力！

## 0. 准备工作

In [1]:
from IPython.display import display, Image
from openai import OpenAI
import os
import pandas as pd
import json
import io
from PIL import Image
import requests

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

#Lets import some helper functions for assistants from https://cookbook.openai.com/examples/assistants_api_overview_python
def show_json(obj):
    display(json.loads(obj.model_dump_json()))

def submit_message(assistant_id, thread, user_message,file_ids=None):
    params = {
        'thread_id': thread.id,
        'role': 'user',
        'content': user_message,
    }
    if file_ids:
        params['file_ids']=file_ids

    client.beta.threads.messages.create(
        **params
)
    return client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant_id,
)

def get_response(thread):
    return client.beta.threads.messages.list(thread_id=thread.id)


## 1. 创建内容

在本示例中，我们将为虚构公司 NotReal Corporation 的季度财务回顾制作一份简短的演示文稿。我们希望突出一些正在影响公司盈利能力的关键趋势。<br> 假设我们手头有一些财务数据。让我们加载数据并看一看……

In [2]:
financial_data_path = 'data/NotRealCorp_financial_data.json'
financial_data = pd.read_json(financial_data_path)
financial_data.head(5)


,Year,Quarter,Distribution channel,Revenue ($M),Costs ($M),Customer count,Time
0,2021,Q1,Online Sales,1.50,1.301953,150,2021 Q1
1,2021,Q1,Direct Sales,1.50,1.380809,151,2021 Q1
2,2021,Q1,Retail Partners,1.50,1.348246,152,2021 Q1
3,2021,Q2,Online Sales,1.52,1.308608,152,2021 Q2
4,2021,Q2,Direct Sales,1.52,1.413305,153,2021 Q2


如你所见，这些数据包含不同分销渠道的季度收入、成本和客户数据。让我们创建一个 Assistant，
让它充当个人分析师，为我们的 PowerPoint 制作一张漂亮的可视化图表！

首先，我们需要上传文件，以便 Assistant 可以访问它。

In [3]:
file = client.files.create(
  file=open('data/NotRealCorp_financial_data.json',"rb"),
  purpose='assistants',
)


现在，我们可以创建 Assistant 了。我们可以指示 Assistant 充当数据科学家，接收我们提出的任何查询，并运行必要的代码来输出合适的数据可视化。这里的 instructions 参数类似于 Chat Completions API 中的系统指令，可以帮助引导 Assistant。我们还可以启用 Code Interpreter 工具，让 Assistant 能够编写代码。最后，我们可以指定想要使用的文件，在本例中就是我们上面创建的 `financial_data` 文件。

In [4]:
assistant = client.beta.assistants.create(
  instructions="You are a data scientist assistant. When given data and a query, write the proper code and create the proper visualization",
  model="gpt-4-1106-preview",
  tools=[{"type": "code_interpreter"}],
  file_ids=[file.id]
)


现在让我们创建一个线程（thread），作为第一个请求，让 Assistant 计算季度利润，然后按分销渠道绘制利润随时间变化的图表。Assistant 会自动计算每个季度的利润，并创建一个将季度和年份组合起来的新列，而无需我们直接提出要求。我们还可以指定每条线的颜色。

In [5]:
thread = client.beta.threads.create(
  messages=[
    {
      "role": "user",
      "content": "Calculate profit (revenue minus cost) by quarter and year, and visualize as a line plot across the distribution channels, where the colors of the lines are green, light red, and light blue",
      "file_ids": [file.id]
    }
  ]
)


现在我们可以执行线程的运行（run）了

In [6]:

run = client.beta.threads.runs.create(
    thread_id=thread.id,
    assistant_id=assistant.id,
)


现在我们可以启动一个循环，检查图像是否已创建。注意：这可能需要几分钟

In [7]:
messages = client.beta.threads.messages.list(thread_id=thread.id)


In [8]:
import time

while True:
    messages = client.beta.threads.messages.list(thread_id=thread.id)
    try:
        #See if image has been created
        messages.data[0].content[0].image_file
        #Sleep to make sure run has completed
        time.sleep(5)
        print('Plot created!')
        break
    except:
        time.sleep(10)
        print('Assistant still working...')


Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Assistant still working...
Plot created!


让我们看看 Assistant 添加的消息。

In [9]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
[message.content[0] for message in messages.data]


[MessageContentImageFile(image_file=ImageFile(file_id='file-0rKABLygI02MgwwhpgWdRFY1'), type='image_file'),
 MessageContentText(text=Text(annotations=[], value="The profit has been calculated for each distribution channel by quarter and year. Next, I'll create a line plot to visualize these profits. As specified, I will use green for the 'Online Sales', light red for 'Direct Sales', and light blue for 'Retail Partners' channels. Let's create the plot."), type='text'),
 MessageContentText(text=Text(annotations=[], value="The JSON data has been successfully restructured into a tabular dataframe format. It includes the year, quarter, distribution channel, revenue, costs, customer count, and a combined 'Time' representation of 'Year Quarter'. Now, we have the necessary information to calculate the profit (revenue minus cost) by quarter and year.\n\nTo visualize the profit across the different distribution channels with a line plot, we will proceed with the following steps:\n\n1. Calculate 

我们可以看到，Assistant 的最后一条消息（最新消息显示在最前面）包含了我们要找的图像文件。值得注意的是，由于第一次解析失败，Assistant 多次尝试解析 JSON 数据，这展示了 Assistant 的适应能力。

In [10]:
# Quick helper function to convert our output file to a png
def convert_file_to_png(file_id, write_path):
    data = client.files.content(file_id)
    data_bytes = data.read()
    with open(write_path, "wb") as file:
        file.write(data_bytes)


In [11]:
plot_file_id = messages.data[0].content[0].image_file.file_id
image_path = "../images/NotRealCorp_chart.png"
convert_file_to_png(plot_file_id,image_path)

#Upload
plot_file = client.files.create(
  file=open(image_path, "rb"),
  purpose='assistants'
)


让我们加载图表！

![图像](../images/NotRealCorp_chart.png)

很好！只用一句话，我们就让 Assistant 使用 Code Interpreter
计算出了盈利能力，并绘制了各分销渠道的三条折线图。<br><br>
现在我们的幻灯片有了一张漂亮的图表，但我们还希望配上一些洞察。

## 2. 生成洞察

要从图像中获取洞察，我们只需向线程添加一条新消息。Assistant 会利用消息历史，根据提供的图表为我们给出一些简洁的要点。 

In [12]:
submit_message(assistant.id,thread,"Give me two medium length sentences (~20-30 words per sentence) of the \
      most important insights from the plot you just created.\
             These will be used for a slide deck, and they should be about the\
                     'so what' behind the data."
)


Run(id='run_NWoygMcBfHUr58fCE4Cn6rxN', assistant_id='asst_3T362kLlTyAq0FUnkvjjQczO', cancelled_at=None, completed_at=None, created_at=1701827074, expires_at=1701827674, failed_at=None, file_ids=['file-piTokyHGllwGITzIpoG8dok3'], instructions='You are a data scientist assistant. When given data and a query, write the proper code and create the proper visualization', last_error=None, metadata={}, model='gpt-4-1106-preview', object='thread.run', required_action=None, started_at=None, status='queued', thread_id='thread_73TgtFoJMlEJvb13ngjTnAo3', tools=[ToolAssistantToolsCode(type='code_interpreter')])

现在，运行完成后，我们就可以查看最新消息了

In [13]:
# Hard coded wait for a response, as the assistant may iterate on the bullets.
time.sleep(10)
response = get_response(thread)
bullet_points = response.data[0].content[0].text.value
print(bullet_points)


The plot reveals a consistent upward trend in profits for all distribution channels, indicating successful business growth over time. Particularly, 'Online Sales' shows a notable increase, underscoring the importance of digital presence in revenue generation.


太棒了！Assistant 能够识别出线上销售利润的显著增长，并推断出这表明强大的数字化存在的重要性。现在让我们为这张幻灯片取一个吸引人的标题。

In [14]:
submit_message(assistant.id,thread,"Given the plot and bullet points you created,\
 come up with a very brief title for a slide. It should reflect just the main insights you came up with."
)


Run(id='run_q6E85J31jCw3QkHpjJKl969P', assistant_id='asst_3T362kLlTyAq0FUnkvjjQczO', cancelled_at=None, completed_at=None, created_at=1701827084, expires_at=1701827684, failed_at=None, file_ids=['file-piTokyHGllwGITzIpoG8dok3'], instructions='You are a data scientist assistant. When given data and a query, write the proper code and create the proper visualization', last_error=None, metadata={}, model='gpt-4-1106-preview', object='thread.run', required_action=None, started_at=None, status='queued', thread_id='thread_73TgtFoJMlEJvb13ngjTnAo3', tools=[ToolAssistantToolsCode(type='code_interpreter')])

标题是：

In [15]:
#Wait as assistant may take a few steps
time.sleep(10)
response = get_response(thread)
title = response.data[0].content[0].text.value
print(title)


"Ascending Profits & Digital Dominance"


## 3. DALL·E-3 标题图像

很好，现在我们有了一个标题、一张图表和两个要点。我们几乎可以把这些都放到幻灯片上了，不过作为最后一步，让我们请 DALL·E-3 生成一张图像，用作演示文稿的标题页。<br><br>
*注意：* DALL·E-3 目前尚未在 Assistants API 中提供，但即将推出！<br> <br>
我们将输入公司（NotRealCorp）的简要描述，剩下的交给 DALL·E-3！

In [16]:
company_summary = "NotReal Corp is a prominent hardware company that manufactures and sells processors, graphics cards and other essential computer hardware."


In [17]:
response = client.images.generate(
  model='dall-e-3',
  prompt=f"given this company summary {company_summary}, create an inspirational \
    photo showing the growth and path forward. This will be used at a quarterly\
       financial planning meeting",
       size="1024x1024",
       quality="hd",
       n=1
)
image_url = response.data[0].url


很好，现在我们可以将这张图像添加到线程中。首先，我们可以将图像保存到本地，然后使用 `File` 上传端点将其上传到 Assistants API。我们也来看一下这张图像

In [18]:
dalle_img_path = '../images/dalle_image.png'
img = requests.get(image_url)

#Save locally
with open(dalle_img_path,'wb') as file:
  file.write(img.content)

#Upload
dalle_file = client.files.create(
  file=open(dalle_img_path, "rb"),
  purpose='assistants'
)


  
![图像](../images/dalle_image.png)

## 4. 创建幻灯片

现在我们已经拥有创建幻灯片所需的全部内容。我们可以简单地添加一条消息来请求幻灯片，但这里我们改为给 Assistant 提供一个配合 `python-pptx` 库使用的幻灯片模板。这将确保我们得到想要风格的演示文稿。关于创建模板的说明，请参阅 notebook 末尾的 `Extensions` 部分。

In [19]:
title_template = """
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_PARAGRAPH_ALIGNMENT
from pptx.dml.color import RGBColor

# Create a new presentation object
prs = Presentation()

# Add a blank slide layout
blank_slide_layout = prs.slide_layouts[6]
slide = prs.slides.add_slide(blank_slide_layout)

# Set the background color of the slide to black
background = slide.background
fill = background.fill
fill.solid()
fill.fore_color.rgb = RGBColor(0, 0, 0)

# Add image to the left side of the slide with a margin at the top and bottom
left = Inches(0)
top = Inches(0)
height = prs.slide_height
width = prs.slide_width * 3/5
pic = slide.shapes.add_picture(image_path, left, top, width=width, height=height)

# Add title text box positioned higher
left = prs.slide_width * 3/5
top = Inches(2)
width = prs.slide_width * 2/5
height = Inches(1)
title_box = slide.shapes.add_textbox(left, top, width, height)
title_frame = title_box.text_frame
title_p = title_frame.add_paragraph()
title_p.text = title_text
title_p.font.bold = True
title_p.font.size = Pt(38)
title_p.font.color.rgb = RGBColor(255, 255, 255)
title_p.alignment = PP_PARAGRAPH_ALIGNMENT.CENTER

# Add subtitle text box
left = prs.slide_width * 3/5
top = Inches(3)
width = prs.slide_width * 2/5
height = Inches(1)
subtitle_box = slide.shapes.add_textbox(left, top, width, height)
subtitle_frame = subtitle_box.text_frame
subtitle_p = subtitle_frame.add_paragraph()
subtitle_p.text = subtitle_text
subtitle_p.font.size = Pt(22)
subtitle_p.font.color.rgb = RGBColor(255, 255, 255)
subtitle_p.alignment = PP_PARAGRAPH_ALIGNMENT.CENTER
"""

data_vis_template = """
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_PARAGRAPH_ALIGNMENT
from pptx.dml.color import RGBColor

# Create a new presentation object
prs = Presentation()

# Add a blank slide layout
blank_slide_layout = prs.slide_layouts[6]
slide = prs.slides.add_slide(blank_slide_layout)

# Set the background color of the slide to black
background = slide.background
fill = background.fill
fill.solid()
fill.fore_color.rgb = RGBColor(0, 0, 0)

# Define placeholders
image_path = data_vis_img
title_text = "Maximizing Profits: The Dominance of Online Sales & Direct Sales Optimization"
bullet_points = "• Online Sales consistently lead in profitability across quarters, indicating a strong digital market presence.\n• Direct Sales show fluctuations, suggesting variable performance and the need for targeted improvements in that channel."

# Add image placeholder on the left side of the slide
left = Inches(0.2)
top = Inches(1.8)
height = prs.slide_height - Inches(3)
width = prs.slide_width * 3/5
pic = slide.shapes.add_picture(image_path, left, top, width=width, height=height)

# Add title text spanning the whole width
left = Inches(0)
top = Inches(0)
width = prs.slide_width
height = Inches(1)
title_box = slide.shapes.add_textbox(left, top, width, height)
title_frame = title_box.text_frame
title_frame.margin_top = Inches(0.1)
title_p = title_frame.add_paragraph()
title_p.text = title_text
title_p.font.bold = True
title_p.font.size = Pt(28)
title_p.font.color.rgb = RGBColor(255, 255, 255)
title_p.alignment = PP_PARAGRAPH_ALIGNMENT.CENTER

# Add hardcoded "Key Insights" text and bullet points
left = prs.slide_width * 2/3
top = Inches(1.5)
width = prs.slide_width * 1/3
height = Inches(4.5)
insights_box = slide.shapes.add_textbox(left, top, width, height)
insights_frame = insights_box.text_frame
insights_p = insights_frame.add_paragraph()
insights_p.text = "Key Insights:"
insights_p.font.bold = True
insights_p.font.size = Pt(24)
insights_p.font.color.rgb = RGBColor(0, 128, 100)
insights_p.alignment = PP_PARAGRAPH_ALIGNMENT.LEFT
insights_frame.add_paragraph()


bullet_p = insights_frame.add_paragraph()
bullet_p.text = bullet_points
bullet_p.font.size = Pt(12)
bullet_p.font.color.rgb = RGBColor(255, 255, 255)
bullet_p.line_spacing = 1.5
"""



让我们为幻灯片设置几个变量。我们希望公司名称 NotRealCorp 出现在标题页上，演示文稿的标题应为 'Quarterly financial planning meeting, Q3, 2023'。

In [20]:
title_text = "NotRealCorp"
subtitle_text = "Quarterly financial planning meeting, Q3 2023"


对于数据页，我们有：

这里我们有一个用于创建标题页的模板。下面的模板是通过将理想标题页的图像上传给 GPT-V，并请求生成创建该模板的 `python-pptx` 代码而得到的。模板的输入是 image_path、title_text 和 subtitle_text。

In [21]:
submit_message(assistant.id,thread,f"Use the included code template to create a PPTX slide that follows the template format, but uses the image, company name/title, and document name/subtitle included:\
{title_template}. IMPORTANT: Use the image file included in this message as the image_path image in this first slide, and use the Company Name {title_text} as the title_text variable, and \
  use the subtitle_text {subtitle_text} a the subtitle_text variable. \
    NEST, create a SECOND slide using the following code template: {data_vis_template} to create a PPTX slide that follows the template format, but uses the company name/title, and document name/subtitle included:\
{data_vis_template}. IMPORTANT: Use the line plot image, that is the second attached image in this message, that you created earlier in the thread as the data_vis_img image, and use the data visualization title that you created earlier for the variable title_text, and\
  the bullet points of insights you created earlier for the bullet_points variable. Output these TWO SLIDES as a .pptx file. Make sure the output is two slides, with each slide matching the respective template given in this message.",
              file_ids=[dalle_file.id, plot_file.id]
)


Run(id='run_taLrnOnlDhoywgQFFBOLPlg0', assistant_id='asst_3T362kLlTyAq0FUnkvjjQczO', cancelled_at=None, completed_at=None, created_at=1701827118, expires_at=1701827718, failed_at=None, file_ids=['file-piTokyHGllwGITzIpoG8dok3'], instructions='You are a data scientist assistant. When given data and a query, write the proper code and create the proper visualization', last_error=None, metadata={}, model='gpt-4-1106-preview', object='thread.run', required_action=None, started_at=None, status='queued', thread_id='thread_73TgtFoJMlEJvb13ngjTnAo3', tools=[ToolAssistantToolsCode(type='code_interpreter')])

In [22]:
#May take 1-3 mins
while True:
    try:
        response = get_response(thread)
        pptx_id = response.data[0].content[0].text.annotations[0].file_path.file_id
        print("Successfully retrieved pptx_id:", pptx_id)
        break
    except Exception as e:
        print("Assistant still working on PPTX...")
        time.sleep(10)


Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Assistant still working on PPTX...
Successfully retrieved pptx_id: file-oa0i63qPH4IaJXYj90aA6L4Q


In [25]:
pptx_id = response.data[0].content[0].text.annotations[0].file_path.file_id
ppt_file= client.files.content(pptx_id)
file_obj = io.BytesIO(ppt_file.read())
with open("data/created_slides.pptx", "wb") as f:
    f.write(file_obj.getbuffer())


现在，我们已经保存了一个包含所有已创建内容的 PPTX 文件！<br>

让我们看看仅使用 Assistants API 和 DALL·E-3 创建的 .pptx 截图。Assistants API 目前还没有 `seed` 参数，因此由于 LLM 的非确定性，你运行本 notebook 时看到的 DALL·E-3 图像和措辞会略有不同，但输出在方向上应该是一致的。

标题页：

![标题页](../images/title_slide.png)

以及数据页：

![数据页](../images/data_vis_slide.png)

## 5. 结论

太棒了！虽然这些幻灯片在格式上还可以再调整一下，但我们已经利用 Assistants API、GPT-4 和 DALL·E-3 制作出了很棒的内容。我们拿着一个包含财务数据的 `.csv` 文件，让 Assistant 按分销渠道计算各季度利润、绘制结果图表、从可视化中识别洞察和关键要点，并生成一个概括性标题。而且，仅凭对公司 NotRealCorp 的一段描述，我们就用 DALL·E-3 制作了一张出色的标题图像。<br><br>
虽然要在没有人工参与的情况下完全自动化这一流程还有一段路要走，但希望本 notebook 能让你的幻灯片制作过程更轻松一些。更重要的是，本 notebook 有望让你一窥 Assistants API 的潜力！我们期待看到你的作品。

## 6. 扩展

- 当 DALL·E-3 被整合进 Assistants API 后，我们将能够在线程中请求生成标题图像。 
- GPT-4-Vision 目前尚未在 Assistants API 中提供支持，否则可以用它从折线图图像中提取洞察。
- 本示例中包含的 `python-pptx` 模板是使用 GPT-4-Vision 生成的，因此一个可能的扩展项目是演示将图像转换为幻灯片模板的最佳实践。